# CODIGO DEFINITIVO FEITO POR MIM

In [37]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.vlr.gg/596408/mibr-vs-furia-vct-2026-americas-kickoff-ubf"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

lista = [0, 2, 3, 4, 5]  # índices dos mapas

dfs = []  # lista para armazenar DataFrames de cada mapa

for i in lista:
    
    data = []  # reinicia para cada mapa
    
    all_maps_block = soup.select(".vm-stats-game")[i]
    tables = all_maps_block.select(".wf-table-inset")
    
    for table in tables:
        rows = table.find_all("tr")
        for row in rows:
            cells = [c.get_text(strip=True) for c in row.find_all(["th","td"])]
            if cells:
                data.append(cells)
    
    df = pd.DataFrame(data)
    
    # Remove linhas indesejadas (ajuste se necessário)
    df = df.drop(index=[0, 6]).reset_index(drop=True)
    
    # Renomeia colunas
    df.columns = [
        "Jogador", "", "R", "ACS", "K", "D", "A",
        "+/-", "KAST", "ADR", "HS%", "FK", "FD", "+/-_2"
    ]
    
    df = df.drop(columns=[""])
    
    # Formatação
    df["R"] = df["R"].str[:4]
    df["ACS"] = df["ACS"].str[:3]
    df["K"] = df["K"].str[:2]
    df["D"] = df["D"].str[1:3]
    df["A"] = df["A"].str[:2]
    df["+/-"] = df["+/-"].str.extract(r'([-+]?\d+)').astype(int)
    df["KAST"] = df["KAST"].str[:3]
    df["ADR"] = df["ADR"].str[:3]
    df["HS%"] = df["HS%"].str[:3]
    df["FK"] = df["FK"].str.extract(r'(\d+)').astype(int)
    df["FD"] = df["FD"].str.extract(r'(\d+)').astype(int)
    df["+/-_2"] = df["+/-_2"].str.extract(r'([-+]?\d+)').astype(int)

    # Converte tipos
    df["R"] = df["R"].astype(str).str.extract(r'(\d+\.\d+)').astype(float)
    df["ACS"] = df["ACS"].astype(str).str.extract(r'^(\d{2,3})').astype(int)
    df["K"] = df["K"].astype(str).str.extract(r'^(\d{1,2})').astype(int)
    df["D"] = df["D"].astype(str).str.extract(r'^(\d{1,2})').astype(int)
    df["A"] = df["A"].astype(str).str.extract(r'^(\d{1,2})').astype(int)
    df["+/-"] = df["+/-"].astype(str).str.extract(r'([-+]?\d+)').astype(int)
    df["KAST"] = df["KAST"].astype(str).str.extract(r'^(\d{1,3})').astype(int)
    df["HS%"] = df["HS%"].astype(str).str.extract(r'^(\d{1,3})').astype(int)
    df["ADR"] = df["ADR"].astype(str).str.extract(r'^(\d{2,3})').astype(int)
    df["FK"] = df["FK"].astype(str).str[0].astype(int)
    df["FD"] = df["FD"].astype(str).str[0].astype(int)
    df["+/-_2"] = df["+/-_2"].astype(str).str.extract(r'([-+]?\d+)').astype(int)

    # adiciona coluna do mapa
    df["Mapa"] = i
    
    dfs.append(df)

# Junta todos os mapas
df_final = pd.concat(dfs, ignore_index=True)

# Exporta para CSV
df_final.to_csv("vlr2.csv", index=False)

print("CSV salvo com sucesso!")


CSV salvo com sucesso!


#  CODIGO ADAPTADO E MELHORADO POR IA

# Coleta Automatizada

In [1]:
import re
import sys
import time
import os

import gspread
import pandas as pd
import requests
from bs4 import BeautifulSoup
from google.oauth2.service_account import Credentials

# ─────────────────────────────────────────────
#  CONFIGURAÇÃO
# ─────────────────────────────────────────────

CREDENTIALS_FILE = "teste-coleta-489718-87d9ed749c61.json"
SPREADSHEET_ID   = "1wAcPW4R9jIfi9fkgOO-sPhQGxJQJQxvAp2t5Ff0upMg"
ABA_URLS         = "Página1"
ABA_STATS        = "Stats"
OUTPUT_CSV       = "vlr_stats2.csv"
DELAY            = 3

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# ─────────────────────────────────────────────
#  GOOGLE SHEETS
# ─────────────────────────────────────────────

def conectar_sheets():
    creds  = Credentials.from_service_account_file(CREDENTIALS_FILE, scopes=SCOPES)
    client = gspread.authorize(creds)
    return client.open_by_key(SPREADSHEET_ID)


def ler_urls_pendentes(sheet):
    aba     = sheet.worksheet(ABA_URLS)
    valores = aba.get_all_values()
    pendentes = []
    for i, row in enumerate(valores, start=1):
        if not row or not row[0].strip():
            continue
        url    = row[0].strip()
        status = row[1].strip().lower() if len(row) > 1 else ""
        if status not in ("ok", "erro"):
            pendentes.append((i, url))
    return aba, pendentes


def marcar_status(aba, linha, status):
    aba.update_cell(linha, 2, status)


def _expandir_aba(aba, linhas_necessarias):
    """Expande a aba automaticamente se o número de linhas for insuficiente."""
    props = aba.spreadsheet.fetch_sheet_metadata()
    for s in props["sheets"]:
        if s["properties"]["title"] == aba.title:
            atual = s["properties"]["gridProperties"]["rowCount"]
            if linhas_necessarias > atual:
                aba.add_rows(linhas_necessarias - atual + 1000)
                print(f"     📐 Aba expandida para {linhas_necessarias + 1000} linhas")
            break


def escrever_stats(sheet, df):
    try:
        aba = sheet.worksheet(ABA_STATS)
    except gspread.exceptions.WorksheetNotFound:
        aba = sheet.add_worksheet(title=ABA_STATS, rows=1000, cols=20)

    existentes = aba.get_all_values()
    df_limpo   = df.fillna("")
    novas_rows = df_limpo.values.tolist()

    if not existentes:
        todas = [df_limpo.columns.tolist()] + novas_rows
        _expandir_aba(aba, len(todas))
        aba.update(values=todas, range_name="A1")
    else:
        proxima = len(existentes) + 1
        _expandir_aba(aba, proxima + len(novas_rows))
        aba.update(values=novas_rows, range_name=f"A{proxima}")

    print(f"     ✔ {len(df)} linhas escritas na aba '{ABA_STATS}'")


def salvar_csv(df, caminho):
    existe = os.path.isfile(caminho)
    df.to_csv(caminho, mode="a", index=False, header=not existe)
    print(f"     ✔ {len(df)} linhas salvas em '{caminho}'")


# ─────────────────────────────────────────────
#  SCRAPING
# ─────────────────────────────────────────────

def get_map_name(block):
    header = block.select_one(".vm-stats-game-header")
    if header:
        text  = header.get_text(" ", strip=True)
        match = re.search(
            r'\b(Bind|Haven|Split|Ascent|Icebox|Breeze|Fracture|Pearl|Lotus|Sunset|Abyss|Corrode)\b',
            text, re.IGNORECASE,
        )
        if match:
            return match.group(1).capitalize()
    return "AllMaps"


def bloco_tem_dados(block):
    for row in block.select(".wf-table-inset tr"):
        cells = row.select("td")
        if cells and any(c.get_text(strip=True) for c in cells):
            return True
    return False


def detectar_mapas(soup):
    blocks  = soup.select(".vm-stats-game")
    validos = [i for i, b in enumerate(blocks) if bloco_tem_dados(b)]
    return validos


def mod_both(cell):
    el = cell.select_one(".mod-both")
    return el.get_text(strip=True) if el else cell.get_text(strip=True)


def extrair_jogador(cell):
    nome_el = cell.select_one(".text-of")
    time_el = cell.select_one(".ge-text-light")
    nome = nome_el.get_text(strip=True) if nome_el else ""
    time = time_el.get_text(strip=True) if time_el else ""
    return nome, time


def extrair_agente(cell):
    img = cell.select_one("img")
    if not img:
        return None
    nome = img.get("title") or img.get("alt") or ""
    if nome:
        return nome.strip().capitalize()
    src   = img.get("src", "")
    match = re.search(r'/([^/]+?)(?:_\w+)?\.(?:png|webp|jpg)$', src, re.IGNORECASE)
    return match.group(1).replace("-", " ").capitalize() if match else None


def p_int(text, pattern=r'(\d+)'):
    m = re.search(pattern, text)
    return int(m.group(1)) if m else None

def p_float(text):
    m = re.search(r'(\d+\.\d+)', text)
    return float(m.group(1)) if m else None

def p_signed(text):
    m = re.search(r'([+-]?\d+)', text)
    return int(m.group(1)) if m else None


def extrair_linha(row):
    cells = row.select("td")
    if len(cells) < 13:
        return None

    jogador, time = extrair_jogador(cells[0])
    if not jogador:
        return None

    def v(i):
        return mod_both(cells[i])

    def kda(i):
        m = re.search(r'^(\d+)', v(i).strip())
        return int(m.group(1)) if m else None

    return {
        "Jogador":   jogador,
        "Time":      time,
        "Agente":    extrair_agente(cells[1]),
        "R":         p_float(v(2)),
        "ACS":       p_int(v(3)),
        "K":         kda(4),
        "D":         kda(5),
        "A":         kda(6),
        "+/-":       p_signed(v(7)),
        "KAST":      p_int(v(8)),
        "ADR":       p_int(v(9)),
        "HS%":       p_int(v(10)),
        "FK":        p_int(v(11)),
        "FD":        p_int(v(12)),
        "+/-_FK_FD": p_signed(v(13)) if len(cells) > 13 else None,
    }


def extrair_dataframe(block, indice, nome_mapa, url):
    rows_data = []
    for table in block.select(".wf-table-inset"):
        for row in table.select("tr"):
            if row.select("th") and not row.select("td"):
                continue
            parsed = extrair_linha(row)
            if parsed:
                rows_data.append(parsed)

    if not rows_data:
        return None

    df = pd.DataFrame(rows_data)
    df.insert(0, "URL",      url)
    df.insert(1, "Mapa_Num", indice)
    df.insert(2, "Mapa",     nome_mapa)
    return df


def scrape_url(url):
    headers  = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    soup       = BeautifulSoup(response.text, "html.parser")
    all_blocks = soup.select(".vm-stats-game")

    if not all_blocks:
        return None

    indices = detectar_mapas(soup)
    if not indices:
        return None

    dfs = []
    for i in indices:
        nome = get_map_name(all_blocks[i])
        tipo = "All Maps" if nome == "AllMaps" else f"Mapa {i}"
        print(f"       [{tipo}] {nome}", end=" ... ")
        df = extrair_dataframe(all_blocks[i], i, nome, url)
        if df is not None:
            dfs.append(df)
            print(f"{len(df)} jogadores")
        else:
            print("sem dados")

    return pd.concat(dfs, ignore_index=True) if dfs else None


# ─────────────────────────────────────────────
#  EXECUÇÃO PRINCIPAL
# ─────────────────────────────────────────────

def main():
    print("🔗 Conectando ao Google Sheets...")
    sheet = conectar_sheets()

    aba_urls, pendentes = ler_urls_pendentes(sheet)

    if not pendentes:
        print("✅ Nenhuma URL pendente. Todas já foram coletadas.")
        return

    print(f"📋 {len(pendentes)} URL(s) pendente(s)\n")

    for linha, url in pendentes:
        print(f"  → Linha {linha}: {url}")
        try:
            df = scrape_url(url)
            if df is not None and not df.empty:
                escrever_stats(sheet, df)
                salvar_csv(df, OUTPUT_CSV)
                marcar_status(aba_urls, linha, "ok")
            else:
                print("     ⚠️  Sem dados (jogo não aconteceu ainda?)")
                marcar_status(aba_urls, linha, "erro")
        except Exception as e:
            print(f"     ❌ Erro: {e}")
            marcar_status(aba_urls, linha, "erro")

        time.sleep(DELAY)

    print(f"\n✅ Coleta finalizada! Dados salvos em '{OUTPUT_CSV}' e na planilha.")


if __name__ == "__main__":
    main()

🔗 Conectando ao Google Sheets...
📋 312 URL(s) pendente(s)

  → Linha 2: https://www.vlr.gg/596398/loud-vs-cloud9-vct-2026-americas-kickoff-ur1
       [Mapa 0] Haven ... 10 jogadores
       [All Maps] AllMaps ... 10 jogadores
       [Mapa 2] Abyss ... 10 jogadores
     ✔ 30 linhas escritas na aba 'Stats'
     ✔ 30 linhas salvas em 'vlr_stats2.csv'
  → Linha 3: https://www.vlr.gg/596399/envy-vs-evil-geniuses-vct-2026-americas-kickoff-ur1
       [Mapa 0] Abyss ... 10 jogadores
       [All Maps] AllMaps ... 10 jogadores
       [Mapa 2] Corrode ... 10 jogadores
       [Mapa 3] Bind ... 10 jogadores
     ✔ 40 linhas escritas na aba 'Stats'
     ✔ 40 linhas salvas em 'vlr_stats2.csv'
  → Linha 4: https://www.vlr.gg/596400/kr-esports-vs-furia-vct-2026-americas-kickoff-ur1
       [Mapa 0] Pearl ... 10 jogadores
       [All Maps] AllMaps ... 10 jogadores
       [Mapa 2] Breeze ... 10 jogadores
       [Mapa 3] Abyss ... 10 jogadores
     ✔ 40 linhas escritas na aba 'Stats'
     ✔ 40 linhas salvas

/tmp/ipykernel_31043/1992354590.py:247: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


     ✔ 40 linhas escritas na aba 'Stats'
     ✔ 40 linhas salvas em 'vlr_stats2.csv'
  → Linha 279: https://www.vlr.gg/183796/mibr-vs-sentinels-champions-tour-2023-americas-league-w4
       [Mapa 0] Pearl ... 10 jogadores
       [All Maps] AllMaps ... 10 jogadores
       [Mapa 2] Fracture ... 10 jogadores
     ✔ 30 linhas escritas na aba 'Stats'
     ✔ 30 linhas salvas em 'vlr_stats2.csv'
  → Linha 280: https://www.vlr.gg/183797/leviat-n-vs-100-thieves-champions-tour-2023-americas-league-w4
       [Mapa 0] Icebox ... 10 jogadores
       [All Maps] AllMaps ... 10 jogadores
       [Mapa 2] Haven ... 10 jogadores
       [Mapa 3] Lotus ... 10 jogadores
     ✔ 40 linhas escritas na aba 'Stats'
     ✔ 40 linhas salvas em 'vlr_stats2.csv'
  → Linha 281: https://www.vlr.gg/183798/evil-geniuses-vs-loud-champions-tour-2023-americas-league-w4
       [Mapa 0] Split ... 10 jogadores
       [All Maps] AllMaps ... 10 jogadores
       [Mapa 2] Haven ... 10 jogadores
       [Mapa 3] Pearl ... 10 jogado

# COLETA DOS PLACARES DOS JOGOS

In [2]:
import re
import time
import os

import gspread
import pandas as pd
import requests
from bs4 import BeautifulSoup
from google.oauth2.service_account import Credentials

# ─────────────────────────────────────────────
#  CONFIGURAÇÃO
# ─────────────────────────────────────────────

CREDENTIALS_FILE = "teste-coleta-489718-87d9ed749c61.json"
SPREADSHEET_ID   = "1wAcPW4R9jIfi9fkgOO-sPhQGxJQJQxvAp2t5Ff0upMg"
ABA_URLS         = "Página1"
ABA_PLACAR       = "Placar"
OUTPUT_CSV       = "vlr_placar2.csv"
DELAY            = 2

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# ─────────────────────────────────────────────
#  GOOGLE SHEETS
# ─────────────────────────────────────────────

def conectar_sheets():
    creds  = Credentials.from_service_account_file(CREDENTIALS_FILE, scopes=SCOPES)
    client = gspread.authorize(creds)
    return client.open_by_key(SPREADSHEET_ID)


def ler_urls_pendentes(sheet):
    aba     = sheet.worksheet(ABA_URLS)
    valores = aba.get_all_values()
    pendentes = []
    for i, row in enumerate(valores, start=1):
        if not row or not row[0].strip():
            continue
        url    = row[0].strip()
        status = row[2].strip().lower() if len(row) > 2 else ""
        if status not in ("ok", "erro"):
            pendentes.append((i, url))
    return aba, pendentes


def marcar_status(aba, linha, status):
    aba.update_cell(linha, 3, status)


def _expandir_aba(aba, linhas_necessarias):
    """Expande a aba automaticamente se o número de linhas for insuficiente."""
    props = aba.spreadsheet.fetch_sheet_metadata()
    for s in props["sheets"]:
        if s["properties"]["title"] == aba.title:
            atual = s["properties"]["gridProperties"]["rowCount"]
            if linhas_necessarias > atual:
                aba.add_rows(linhas_necessarias - atual + 1000)
                print(f"     📐 Aba expandida para {linhas_necessarias + 1000} linhas")
            break


def escrever_placar(sheet, df):
    try:
        aba = sheet.worksheet(ABA_PLACAR)
    except gspread.exceptions.WorksheetNotFound:
        aba = sheet.add_worksheet(title=ABA_PLACAR, rows=100, cols=10)

    existentes = aba.get_all_values()
    df_limpo   = df.fillna("")
    novas_rows = df_limpo.values.tolist()

    if not existentes:
        todas = [df_limpo.columns.tolist()] + novas_rows
        _expandir_aba(aba, len(todas))
        aba.update(values=todas, range_name="A1")
    else:
        proxima = len(existentes) + 1
        _expandir_aba(aba, proxima + len(novas_rows))
        aba.update(values=novas_rows, range_name=f"A{proxima}")

    print(f"     ✔ {len(df)} linhas escritas na aba '{ABA_PLACAR}'")


def salvar_csv(df, caminho):
    existe = os.path.isfile(caminho)
    df.to_csv(caminho, mode="a", index=False, header=not existe)
    print(f"     ✔ {len(df)} linhas salvas em '{caminho}'")


# ─────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────

def extrair_times_da_pagina(soup):
    """Pega os nomes dos dois times do topo da página."""
    els = soup.select(".match-header-link-name")
    if len(els) >= 2:
        return els[0].get_text(strip=True), els[1].get_text(strip=True)
    return None, None


def get_map_name(block):
    """
    Retorna nome do mapa se for bloco individual (Bind, Haven...),
    ou None se for o bloco All Maps.
    """
    header = block.select_one(".vm-stats-game-header")
    if not header:
        return None
    text  = header.get_text(" ", strip=True)
    match = re.search(
        r'\b(Bind|Haven|Split|Ascent|Icebox|Breeze|Fracture|Pearl|Lotus|Sunset|Abyss|Corrode)\b',
        text, re.IGNORECASE,
    )
    return match.group(1).capitalize() if match else None


def bloco_tem_dados(block):
    """Retorna True se o bloco tiver linhas de jogadores preenchidas."""
    for row in block.select(".wf-table-inset tr"):
        cells = row.select("td")
        if cells and any(c.get_text(strip=True) for c in cells):
            return True
    return False


def extrair_score_bloco(block):
    """
    Extrai os dois scores totais do header de um bloco de mapa individual.
    Retorna (score_time1, score_time2).
    """
    header = block.select_one(".vm-stats-game-header")
    if not header:
        return None, None

    score_els = [
        el for el in header.select(".score")
        if "mod-t"  not in el.get("class", [])
        and "mod-ct" not in el.get("class", [])
    ]

    def to_int(el):
        try:
            return int(el.get_text(strip=True))
        except (ValueError, AttributeError):
            return None

    s1 = to_int(score_els[0]) if len(score_els) > 0 else None
    s2 = to_int(score_els[1]) if len(score_els) > 1 else None
    return s1, s2


# ─────────────────────────────────────────────
#  SCRAPING
# ─────────────────────────────────────────────

def scrape_placar(url):
    req_headers = {"User-Agent": "Mozilla/5.0"}
    response    = requests.get(url, headers=req_headers, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    time1, time2 = extrair_times_da_pagina(soup)
    if not time1 or not time2:
        print("     ⚠️  Não encontrou os times na página")
        return None

    all_blocks = soup.select(".vm-stats-game")
    if not all_blocks:
        return None

    rows = []
    for block in all_blocks:
        nome_mapa = get_map_name(block)

        if not nome_mapa or not bloco_tem_dados(block):
            continue

        s1, s2 = extrair_score_bloco(block)

        if s1 is not None and s2 is not None:
            vencedor = time1 if s1 > s2 else (time2 if s2 > s1 else "Empate")
        else:
            vencedor = None

        rows.append({
            "URL":      url,
            "Mapa":     nome_mapa,
            "Time1":    time1,
            "Score_T1": s1,
            "Time2":    time2,
            "Score_T2": s2,
            "Vencedor": vencedor,
        })

        print(f"       {nome_mapa}: {time1} {s1} x {s2} {time2}  → {vencedor}")

    if not rows:
        return None

    df_mapas = pd.DataFrame(rows)

    # Vencedor = quem ganhou mais mapas individuais
    mapas_t1 = (df_mapas["Vencedor"] == time1).sum()
    mapas_t2 = (df_mapas["Vencedor"] == time2).sum()

    if mapas_t1 > mapas_t2:
        venc_all = time1
    elif mapas_t2 > mapas_t1:
        venc_all = time2
    else:
        venc_all = "Empate"

    all_maps_row = pd.DataFrame([{
        "URL":      url,
        "Mapa":     "AllMaps",
        "Time1":    time1,
        "Mapas_T1": int(mapas_t1),
        "Score_T1": int(df_mapas["Score_T1"].sum()),
        "Time2":    time2,
        "Mapas_T2": int(mapas_t2),
        "Score_T2": int(df_mapas["Score_T2"].sum()),
        "Vencedor": venc_all,
    }])

    print(f"       AllMaps: {time1} {mapas_t1} x {mapas_t2} {time2}  → {venc_all}")

    df_mapas["Mapas_T1"] = None
    df_mapas["Mapas_T2"] = None

    cols = ["URL", "Mapa", "Time1", "Mapas_T1", "Score_T1",
            "Time2", "Mapas_T2", "Score_T2", "Vencedor"]
    df_mapas     = df_mapas[cols]
    all_maps_row = all_maps_row[cols]

    return pd.concat([df_mapas, all_maps_row], ignore_index=True)


# ─────────────────────────────────────────────
#  EXECUÇÃO PRINCIPAL
# ─────────────────────────────────────────────

def main():
    print("🔗 Conectando ao Google Sheets...")
    sheet = conectar_sheets()

    aba_urls, pendentes = ler_urls_pendentes(sheet)

    if not pendentes:
        print("✅ Nenhuma URL pendente.")
        return

    print(f"📋 {len(pendentes)} URL(s) pendente(s)\n")

    for linha, url in pendentes:
        print(f"  → Linha {linha}: {url}")
        try:
            df = scrape_placar(url)
            if df is not None and not df.empty:
                escrever_placar(sheet, df)
                salvar_csv(df, OUTPUT_CSV)
                marcar_status(aba_urls, linha, "ok")
            else:
                print("     ⚠️  Sem dados")
                marcar_status(aba_urls, linha, "erro")
        except Exception as e:
            print(f"     ❌ Erro: {e}")
            marcar_status(aba_urls, linha, "erro")

        time.sleep(DELAY)

    print(f"\n✅ Finalizado! Dados em '{OUTPUT_CSV}' e na aba '{ABA_PLACAR}'.")


if __name__ == "__main__":
    main()

🔗 Conectando ao Google Sheets...
📋 312 URL(s) pendente(s)

  → Linha 2: https://www.vlr.gg/596398/loud-vs-cloud9-vct-2026-americas-kickoff-ur1
       Haven: LOUD 7 x 13 Cloud9  → Cloud9
       Abyss: LOUD 11 x 13 Cloud9  → Cloud9
       AllMaps: LOUD 0 x 2 Cloud9  → Cloud9
     ✔ 3 linhas escritas na aba 'Placar'
     ✔ 3 linhas salvas em 'vlr_placar2.csv'
  → Linha 3: https://www.vlr.gg/596399/envy-vs-evil-geniuses-vct-2026-americas-kickoff-ur1
       Abyss: ENVY 9 x 13 Evil Geniuses  → Evil Geniuses
       Corrode: ENVY 13 x 10 Evil Geniuses  → ENVY
       Bind: ENVY 13 x 9 Evil Geniuses  → ENVY
       AllMaps: ENVY 2 x 1 Evil Geniuses  → ENVY
     ✔ 4 linhas escritas na aba 'Placar'
     ✔ 4 linhas salvas em 'vlr_placar2.csv'
  → Linha 4: https://www.vlr.gg/596400/kr-esports-vs-furia-vct-2026-americas-kickoff-ur1
       Pearl: KRÜ Esports 13 x 10 FURIA  → KRÜ Esports
       Breeze: KRÜ Esports 8 x 13 FURIA  → FURIA
       Abyss: KRÜ Esports 5 x 13 FURIA  → FURIA
       AllMaps: KRÜ 